# OpenRAN POWDER Testbed — UE Data Analysis
**Setup**: 1 Open5GS core · 1 gNB (50 srsenb, ZMQ) · 49 UEs  
**Radio**: srsRAN 4G · ZMQ · 10 MHz · 50 PRB  

### Effective Throughput
```
effective_tput_mbps = reported_tput_mbps × (1 − packet_loss / 100)
```
ZMQ is a perfect software channel — loss reflects radio scheduler PRB limits, not RF noise.

In [ ]:
!pip install matplotlib numpy pandas seaborn -q
import json, warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib.colors as mcolors
import seaborn as sns
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f9fafb',
    'axes.grid':True,'grid.color':'#e5e7eb','grid.linewidth':0.6,
    'font.size':10,'axes.titlesize':12,'axes.titleweight':'bold',
    'xtick.labelsize':8,'ytick.labelsize':9,'legend.fontsize':9})
BLUE='#3b82d4'; GREEN='#22c55e'; ORANGE='#f97316'; RED='#ef4444'; PURPLE='#7c3aed'
RATES=[1,10,20,50,100,200,300,400,500]
print('imports OK')

In [ ]:
# Embedded dataset — all measurements from POWDER testbed
RAW = {"udp":[[1,1,1.001,27.26,23.576],[1,10,10.0,92.49,17.018],[1,20,19.999,95.54,21.801],[1,50,49.995,98.26,24.661],[1,100,99.991,99.04,19.518],[1,200,199.981,99.44,21.216],[1,300,299.976,98.9,19.273],[1,400,399.982,99.19,23.196],[1,500,499.966,99.01,20.234],[2,300,299.975,99.13,39.783],[4,1,1.001,32.22,23.918],[4,10,10.0,92.95,26.736],[4,20,19.999,95.65,30.249],[4,50,49.995,98.34,26.812],[4,100,99.992,99.07,23.941],[4,200,199.983,99.44,26.677],[4,300,299.983,99.65,28.986],[4,400,399.964,99.68,26.508],[4,500,499.959,99.68,32.178],[5,1,1.001,0.0,13.996],[5,10,10.0,86.11,7.162],[5,20,19.999,92.95,12.637],[5,50,49.995,97.24,7.502],[5,100,99.993,98.54,10.818],[5,200,199.982,99.25,8.198],[5,300,299.983,99.44,13.225],[5,400,399.971,99.59,10.974],[5,500,499.982,99.6,9.718],[6,1,1.001,18.53,20.638],[6,10,10.0,91.84,16.18],[6,20,19.999,95.51,13.995],[6,50,49.996,98.34,16.996],[6,100,99.992,98.51,15.101],[6,200,199.984,98.92,14.025],[6,300,299.982,99.47,14.369],[6,400,399.97,99.01,13.437],[6,500,499.989,99.33,14.355],[7,100,99.991,99.41,26.441],[7,200,199.982,99.62,35.132],[7,300,299.979,99.05,26.043],[7,400,399.98,99.01,25.66],[7,500,499.973,99.45,30.329],[8,1,1.001,31.68,20.941],[8,10,10.0,92.91,24.325],[8,20,19.999,96.06,23.753],[8,50,49.996,98.36,23.192],[8,100,99.994,99.3,25.564],[8,200,199.983,99.5,17.576],[8,300,299.976,99.58,16.911],[8,400,399.977,99.4,28.916],[8,500,499.965,99.56,30.079],[9,1,1.001,0.0,13.817],[9,10,10.0,90.0,10.831],[9,20,19.999,94.77,9.714],[9,50,49.995,97.78,8.979],[9,100,99.991,98.51,12.971],[9,200,199.98,99.38,11.092],[9,300,299.996,99.55,12.296],[9,400,399.965,99.45,12.453],[9,500,499.954,99.62,11.991],[10,1,1.001,52.48,33.916],[10,10,10.0,94.51,27.635],[10,20,19.999,97.56,26.907],[10,50,49.995,98.87,25.899],[10,100,99.99,98.9,29.759],[10,200,199.982,99.09,27.167],[10,300,299.975,99.39,31.15],[10,400,399.961,99.24,35.78],[11,1,1.001,13.9,16.917],[11,10,10.0,91.2,13.11],[11,20,19.999,94.97,12.547],[11,50,49.995,98.21,15.527],[11,100,99.991,98.94,12.705],[11,200,199.984,99.38,11.923],[11,300,299.991,99.64,11.746],[11,400,399.992,99.6,14.52],[11,500,499.992,99.77,12.551],[12,1,1.001,32.44,23.737],[12,10,10.0,93.18,22.052],[12,20,19.999,96.29,25.744],[12,50,49.995,97.92,28.634],[12,100,99.992,99.02,23.297],[12,200,199.985,99.47,28.35],[12,300,299.977,99.45,25.292],[12,400,399.972,99.73,15.491],[12,500,499.955,99.53,27.87],[13,1,1.001,0.0,11.773],[13,10,10.0,76.61,6.418],[13,20,19.999,88.23,3.969],[13,50,49.995,95.27,4.096],[13,100,99.992,97.62,3.897],[13,200,199.984,98.8,4.978],[13,300,299.983,99.18,3.666],[13,400,399.959,99.38,3.909],[13,500,499.963,99.5,4.042],[15,300,299.983,99.07,17.717],[15,400,399.973,98.43,20.52],[15,500,499.99,99.6,18.679],[16,1,1.001,0.0,10.313],[16,10,10.0,79.21,5.066],[16,20,19.999,89.48,4.143],[16,50,49.995,95.65,5.612],[16,100,99.991,97.85,5.008],[16,200,199.986,98.91,5.934],[16,300,299.979,99.18,5.948],[16,400,399.971,99.33,5.597],[16,500,499.963,99.5,5.693],[17,200,199.984,99.35,18.953],[17,300,299.974,99.77,16.639],[17,400,399.973,99.74,20.811],[17,500,499.956,99.69,22.48],[18,1,1.001,32.87,24.634],[18,10,10.0,93.03,28.652],[18,20,19.999,96.18,28.944],[18,50,49.995,98.38,26.192],[18,100,99.99,99.07,19.67],[18,200,199.982,99.56,28.397],[18,300,299.977,99.42,28.237],[18,400,399.961,99.8,28.296],[18,500,499.969,99.82,27.963],[19,400,399.973,98.75,28.598],[19,500,499.957,99.57,29.463],[20,400,399.986,98.73,38.575],[20,500,499.963,98.86,37.651],[21,1,1.001,37.07,21.654],[21,10,10.0,92.83,23.108],[21,20,19.999,96.79,23.519],[21,50,49.995,98.45,26.536],[21,100,99.991,99.22,20.811],[21,200,199.981,99.31,16.06],[21,300,299.976,99.64,26.264],[21,400,399.963,99.58,17.935],[21,500,499.951,99.17,17.819],[22,300,299.991,98.79,22.267],[23,100,99.991,99.32,34.017],[23,200,199.982,99.33,34.812],[23,300,299.974,99.66,47.286],[23,400,399.971,99.67,25.488],[23,500,499.982,99.74,18.853],[26,1,1.001,23.81,26.051],[26,10,10.0,92.13,15.5],[26,20,19.999,96.08,14.798],[26,50,49.995,98.43,13.983],[26,100,99.992,98.65,14.991],[26,200,199.982,99.13,13.592],[26,300,299.976,99.62,14.93],[26,400,399.97,99.49,15.294],[26,500,499.964,99.14,21.106],[27,1,1.001,23.71,20.977],[27,10,10.0,92.29,13.644],[27,20,19.999,95.85,12.303],[27,50,49.995,98.41,15.193],[27,100,99.992,98.89,13.679],[27,200,199.984,99.57,12.661],[27,300,299.986,99.44,14.09],[27,400,399.968,99.69,13.767],[27,500,499.982,99.47,15.201],[28,500,499.946,98.82,34.52],[30,1,1.001,0.0,12.552],[30,10,10.0,89.58,10.32],[30,20,19.999,94.61,9.527],[30,50,49.995,97.83,10.42],[30,100,99.992,98.84,12.583],[30,200,199.983,98.99,11.756],[30,300,299.976,99.57,12.719],[30,400,399.965,99.68,11.99],[30,500,499.949,99.64,12.552],[31,10,10.0,94.06,32.564],[31,20,19.999,96.9,27.985],[31,50,49.996,97.05,29.846],[31,100,99.991,98.31,26.522],[31,200,199.985,97.79,30.311],[31,300,299.971,98.28,35.596],[31,400,399.974,98.56,34.565],[31,500,499.966,99.7,31.887],[32,1,1.001,27.69,22.126],[32,10,10.0,92.1,23.302],[32,20,19.999,96.35,14.815],[32,50,49.995,98.41,20.314],[32,100,99.992,99.1,18.166],[32,200,199.985,99.11,21.389],[32,300,299.968,99.07,22.512],[32,400,399.976,99.6,18.796],[32,500,499.963,99.52,22.104],[33,400,399.969,99.08,25.04],[33,500,499.976,99.1,16.615],[34,1,1.001,0.0,8.698],[34,10,10.0,85.57,7.116],[34,20,19.999,92.47,8.022],[34,50,49.995,96.65,7.54],[34,100,99.992,98.23,6.101],[34,200,199.987,98.84,7.207],[34,300,299.99,99.51,6.166],[34,400,399.973,99.38,8.389],[34,500,499.947,99.59,6.372],[35,1,1.001,32.54,25.847],[35,10,10.0,92.95,28.333],[35,20,19.999,96.55,19.221],[35,50,49.996,98.56,33.639],[35,100,99.993,99.21,35.576],[35,200,199.982,99.41,29.026],[35,300,299.976,99.52,17.75],[35,400,399.959,99.45,22.582],[35,500,499.993,99.65,22.005],[36,1,1.001,12.72,19.193],[36,10,10.0,91.02,14.821],[36,20,19.999,95.62,11.671],[36,50,49.995,97.67,11.257],[36,100,99.991,98.79,12.313],[36,200,199.982,99.34,12.554],[36,300,299.976,99.66,10.322],[36,400,399.975,99.5,11.928],[36,500,499.963,99.66,11.525],[37,1,1.001,3.23,18.628],[37,10,10.0,90.28,13.033],[37,20,19.999,95.17,15.039],[37,50,49.997,97.94,9.811],[37,100,99.991,99.04,14.873],[37,200,199.988,99.36,16.533],[37,300,299.974,99.65,18.439],[37,400,399.964,99.67,11.966],[37,500,499.966,99.55,14.237],[39,1,1.001,23.6,23.176],[39,10,10.0,91.64,14.487],[39,20,19.999,95.87,14.491],[39,50,49.995,98.0,14.895],[39,100,99.992,99.0,14.025],[39,200,199.986,99.49,12.948],[39,300,299.984,99.46,18.699],[39,400,399.96,99.23,12.601],[39,500,499.981,99.63,12.739],[41,1,1.001,27.69,26.671],[41,10,10.0,92.43,16.643],[41,20,19.999,96.1,22.566],[41,50,49.997,97.91,22.544],[41,100,99.992,99.08,23.227],[41,200,199.982,99.33,16.633],[41,300,299.982,99.75,25.152],[41,400,399.973,99.78,23.677],[41,500,499.943,99.2,17.614],[42,1,1.001,24.89,20.914],[42,10,10.0,91.8,15.995],[42,20,19.999,95.72,14.615],[42,50,49.995,98.4,14.616],[42,100,99.992,98.87,14.465],[42,200,199.985,99.31,14.268],[42,300,299.975,99.29,16.172],[42,400,399.969,99.75,14.049],[42,500,499.968,99.73,20.991],[44,1,1.001,46.77,34.352],[44,10,10.0,93.1,29.177],[44,300,299.972,98.9,35.121],[45,200,199.983,99.69,30.313],[45,300,299.978,98.73,26.163],[45,400,399.966,99.43,31.083],[45,500,499.969,99.22,28.317],[46,1,1.001,14.55,19.145],[46,10,10.0,91.28,16.402],[46,20,19.999,95.61,11.325],[46,50,49.995,98.22,15.047],[46,100,99.991,99.07,17.087],[46,200,199.987,99.25,12.569],[46,300,299.99,99.46,19.035],[46,400,399.977,99.59,14.724],[46,500,499.983,99.58,16.147],[47,1,1.001,24.89,21.512],[47,10,10.0,92.21,16.014],[47,20,19.999,96.05,14.665],[47,50,49.997,97.67,15.527],[47,100,99.993,99.12,12.745],[47,200,199.986,99.35,14.828],[47,300,299.977,99.2,15.988],[47,400,399.976,99.57,13.601],[47,500,499.985,99.78,16.125],[49,1,1.001,25.11,20.988],[49,10,10.0,92.22,16.219],[49,20,19.999,96.0,15.47],[49,50,49.995,98.31,14.866],[49,100,99.992,99.08,15.899],[49,200,199.985,99.54,17.029],[49,300,299.978,99.58,14.026],[49,400,399.96,99.62,14.73],[49,500,499.961,99.75,15.467],[50,1,1.001,0.0,15.699],[50,10,10.0,89.87,13.357],[50,20,19.999,94.96,14.239],[50,50,49.996,97.93,10.065],[50,100,99.992,98.9,9.749],[50,200,199.979,99.1,16.235],[50,300,299.982,99.51,12.755],[50,400,399.999,99.73,14.325],[50,500,499.973,99.77,13.988],[2,1,1.001,52.26,33.07],[2,10,10.0,94.13,26.652],[2,20,19.999,97.45,24.806],[2,50,49.995,98.49,23.792],[2,100,99.992,98.06,24.91],[2,200,199.987,99.53,23.824],[2,300,299.971,99.29,26.593],[2,400,399.98,99.56,36.562],[2,500,499.964,99.27,31.361],[3,1,1.001,38.9,30.211],[3,10,10.0,92.94,28.648],[3,20,19.999,96.3,28.169],[3,50,49.995,98.2,28.128],[3,100,99.992,98.65,27.933],[3,200,199.985,99.51,30.727],[3,300,299.98,98.83,32.488],[3,400,399.965,99.49,30.863],[3,500,499.959,99.69,33.906],[14,1,1.001,52.26,32.387],[14,10,10.0,95.09,29.907],[14,20,19.999,97.18,31.57],[14,50,49.995,98.43,27.065],[14,100,99.992,99.49,26.518],[14,200,199.985,99.45,37.227],[14,300,299.979,98.48,25.315],[14,400,399.965,98.68,29.842],[14,500,499.991,99.32,31.302],[15,1,1.001,39.01,32.904],[15,10,10.0,93.14,31.856],[15,20,19.999,96.64,34.126],[15,50,49.996,98.3,29.939],[15,100,99.991,99.07,25.646],[15,200,199.984,99.68,27.145]],"ping":[[1,358.801,787.377,1146.504,227.699],[2,697.549,1547.659,2766.391,559.826],[3,503.529,986.916,1641.907,299.686],[4,444.958,882.986,1316.053,257.619],[5,185.957,345.851,558.63,102.035],[6,327.315,678.909,1008.917,201.849],[7,864.44,1854.51,2752.513,465.122],[8,479.924,920.095,1609.676,293.589],[9,275.368,533.787,826.556,165.155],[10,657.78,1298.899,1934.164,377.621],[11,292.671,633.6,904.614,178.795],[12,402.183,849.738,1269.639,247.747],[13,96.034,210.798,351.586,76.249],[14,675.046,1357.277,2486.698,498.318],[15,631.942,1003.96,1515.057,255.653],[16,125.53,294.809,514.134,103.406],[17,567.585,1160.642,1794.972,371.559],[18,454.224,852.899,1440.428,270.323],[19,769.838,1365.908,1935.329,311.972],[20,855.265,1710.734,2813.133,583.506],[21,541.847,965.829,1394.436,290.183],[22,526.617,958.233,1376.834,290.875],[23,1372.578,2204.651,2854.745,378.191],[24,557.084,1007.858,1451.762,289.036],[25,526.057,978.479,1454.348,296.703],[26,383.457,703.204,1060.431,207.547],[27,377.442,737.608,1415.685,275.12],[28,737.983,1768.07,2679.983,486.05],[29,645.949,1063.053,1505.56,256.623],[30,244.022,520.103,853.775,174.152],[31,897.488,1736.63,2844.977,578.959],[32,405.949,766.466,1275.544,245.76],[33,568.428,1009.393,1457.769,296.248],[34,233.027,447.235,669.847,128.626],[35,460.409,882.615,1281.275,251.462],[36,311.032,641.052,926.943,176.501],[37,235.717,488.099,739.115,147.679],[38,507.151,1063.963,1785.999,315.595],[39,386.727,711.115,1066.35,214.221],[41,471.228,848.811,1387.441,249.275],[42,417.477,723.179,1074.758,205.468],[43,801.787,1561.118,2776.732,555.882],[44,688.459,1543.365,2847.496,605.386],[45,832.889,1657.8,2868.722,557.083],[46,274.861,582.215,1098.995,215.466],[47,362.695,728.01,1060.001,211.942],[48,729.874,1526.003,2712.948,544.771],[49,355.3,731.65,1064.657,210.091],[50,255.466,525.156,789.824,155.405],[2,738.737,1562.695,2279.252,404.287],[3,554.238,1050.805,1539.587,294.684],[14,652.119,1496.149,2706.764,555.798],[15,507.112,998.891,1614.508,292.643]],"ran":[[1,111.8,15.0,0.0,111.4,4.0,517.0,3.0,6.0,2.0,null,null,null,null,null],[2,111.8,15.0,0.0,111.4,4.0,517.0,3.0,6.0,4.0,855580.0,53.0,44.0,30153.8457,22624616.0],[3,111.8,15.0,0.0,null,null,null,null,null,null,855448.0,62.0,46.04,0.0,0.0],[4,111.8,15.0,0.0,null,null,null,null,null,null,855448.0,60.0,48.31,0.0,0.0],[5,111.8,15.0,0.0,null,null,null,null,null,null,null,null,null,null,null],[6,111.8,15.0,0.0,null,null,null,null,null,null,855444.0,68.0,46.55,0.0,0.0],[7,111.8,15.0,0.0,null,null,null,null,null,null,855452.0,62.0,45.55,0.0,0.0],[8,111.8,15.0,0.0,null,null,null,null,null,null,855452.0,66.0,43.71,0.0,0.0],[9,111.8,15.0,0.0,null,null,null,null,null,null,null,null,null,null,null],[10,111.8,15.0,0.0,111.4,4.0,517.0,6.0,12.0,6.0,null,null,null,null,null],[11,111.8,15.0,0.0,null,null,null,null,null,null,null,null,null,null,null],[12,111.8,15.0,0.0,null,null,null,null,null,null,855448.0,48.0,48.0,0.0,0.0],[13,111.8,15.0,0.0,null,null,null,null,null,null,null,null,null,null,null],[14,111.8,15.0,0.0,null,null,null,null,null,null,855448.0,57.0,43.55,0.0,0.0],[15,111.8,15.0,0.0,null,null,null,null,null,null,null,null,null,null,null],[16,111.8,15.0,0.0,null,null,null,null,null,null,null,null,null,null,null],[17,111.8,15.0,0.0,null,null,null,null,null,null,855452.0,70.0,44.67,0.0,0.0],[18,111.8,15.0,0.0,null,null,null,null,null,null,855452.0,74.0,43.96,0.0,0.0],[19,111.8,15.0,0.0,null,null,null,null,null,null,null,null,null,null,null],[20,111.8,15.0,0.0,null,null,null,null,null,null,null,null,null,null,null],[21,111.8,15.0,0.0,null,null,null,null,null,null,null,null,null,null,null],[22,111.8,15.0,0.0,null,null,null,null,null,null,null,null,null,null,null],[23,111.8,15.0,0.0,null,null,null,null,null,null,null,null,null,null,null],[24,111.8,15.0,0.0,null,null,null,null,null,null,null,null,null,null,null],[25,111.8,15.0,0.0,null,null,null,null,null,null,null,null,null,null,null],[26,111.8,15.0,0.0,null,null,null,null,null,null,null,null,null,null,null],[27,111.8,15.0,0.0,null,null,null,null,null,null,855448.0,50.0,45.45,0.0,0.0],[28,111.8,15.0,0.0,null,null,null,null,null,null,null,null,null,null,null],[29,111.8,15.0,0.0,null,null,null,null,null,null,855452.0,null,null,0.0,0.0],[30,111.8,15.0,0.0,null,null,null,null,null,null,855584.0,48.0,43.36,0.0,0.0],[31,111.8,15.0,0.0,null,null,null,null,null,null,855452.0,55.0,55.0,0.0,0.0],[32,111.8,15.0,0.0,null,null,null,null,null,null,855452.0,63.0,43.82,0.0,0.0],[33,111.8,15.0,0.0,null,null,null,null,null,null,855452.0,74.0,50.89,0.0,0.0],[34,111.8,15.0,0.0,null,null,null,null,null,null,855320.0,54.0,48.67,0.0,0.0],[35,111.8,15.0,0.0,null,null,null,null,null,null,null,null,null,null,null],[36,111.8,15.0,0.0,null,null,null,null,null,null,855448.0,48.0,48.0,0.0,0.0],[37,111.8,15.0,0.0,null,null,null,null,null,null,855452.0,52.0,44.0,0.0,0.0],[38,111.8,15.0,0.0,null,null,null,null,null,null,855452.0,null,null,0.0,0.0],[39,111.8,15.0,0.0,null,null,null,null,null,null,855452.0,61.0,43.19,0.0,0.0],[40,null,null,null,null,null,null,null,null,null,null,null,null,null,null],[41,111.8,15.0,0.0,null,null,null,null,null,null,855452.0,null,null,0.0,0.0],[42,111.8,15.0,0.0,null,null,null,null,null,null,855448.0,62.0,62.0,0.0,0.0],[43,111.8,15.0,0.0,null,null,null,null,null,null,null,null,null,null,null],[44,111.8,15.0,0.0,null,null,null,null,null,null,855452.0,62.0,47.0,0.0,0.0],[45,111.8,15.0,0.0,null,null,null,null,null,null,null,null,null,null,null],[46,111.8,15.0,0.0,null,null,null,null,null,null,null,null,null,null,null],[47,111.8,15.0,0.0,null,null,null,null,null,null,null,null,null,null,null],[48,111.8,15.0,0.0,null,null,null,null,null,null,null,null,null,null,null],[49,111.8,15.0,0.0,null,null,null,null,null,null,null,null,null,null,null],[50,111.8,15.0,0.0,null,null,null,null,null,null,null,null,null,null,null]]}

udp_df = pd.DataFrame(RAW['udp'],
    columns=['ue_id','rate_target','tput_mbps','loss_pct','jitter_ms'])
udp_df['effective_mbps'] = udp_df['tput_mbps'] * (1 - udp_df['loss_pct']/100)

ping_df = pd.DataFrame(RAW['ping'],
    columns=['ue_id','ping_min_ms','ping_avg_ms','ping_max_ms','ping_mdev_ms'])

ran_df = pd.DataFrame(RAW['ran'],
    columns=['ue_id','pucch_snr','cqi','ta_us','pusch_snr','pusch_mcs','pusch_tbs',
             'pdsch_prb','prb_util_pct','pdsch_mcs','rss_kB','cpu_max','cpu_mean',
             'dl_brate','ul_brate'])

print(f'UDP rows: {len(udp_df)}  |  UEs: {udp_df.ue_id.nunique()}')
print(f'Ping rows: {len(ping_df)}  |  UEs: {ping_df.ue_id.nunique()}')
print(f'RAN rows: {len(ran_df)}')
display(udp_df.head(3))

In [ ]:
# ── Sent vs Reported vs Effective Throughput ────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(RATES)); w = 0.28
sent  = RATES
recv  = [udp_df[udp_df.rate_target==r]['tput_mbps'].mean() for r in RATES]
eff   = [udp_df[udp_df.rate_target==r]['effective_mbps'].mean() for r in RATES]
ax.bar(x-w, sent, width=w, color='#94a3b8', edgecolor='white', label='Target (sent)')
ax.bar(x,   recv, width=w, color=BLUE,       edgecolor='white', label='Reported')
ax.bar(x+w, eff,  width=w, color=GREEN,       edgecolor='white', label='Effective (after loss)')
for xi, ev in enumerate(eff):
    ax.text(xi+w, ev+1, f'{ev:.1f}', ha='center', fontsize=8, color='#166534', fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels([f'{r}M' for r in RATES])
ax.set_xlabel('Target UDP Rate'); ax.set_ylabel('Throughput (Mbps)')
ax.set_title('Sent vs Reported vs Effective Throughput per Rate')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# ── Loss% vs Rate  +  Reported vs Effective curve ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
mean_loss = [udp_df[udp_df.rate_target==r]['loss_pct'].mean() for r in RATES]
mean_recv = [udp_df[udp_df.rate_target==r]['tput_mbps'].mean() for r in RATES]
mean_eff  = [udp_df[udp_df.rate_target==r]['effective_mbps'].mean() for r in RATES]

ax = axes[0]
ax.plot(RATES, mean_loss, color=RED, marker='o', ms=8, lw=2)
ax.fill_between(RATES, mean_loss, alpha=0.15, color=RED)
for r, l in zip(RATES, mean_loss):
    ax.annotate(f'{l:.0f}%', (r, l), textcoords='offset points', xytext=(0,7),
                ha='center', fontsize=8.5, color='#b91c1c')
ax.set_xscale('log'); ax.set_xticks(RATES)
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.set_xlabel('Target Rate (Mbps)'); ax.set_ylabel('Mean Loss %')
ax.set_title('Packet Loss vs Target Rate'); ax.set_ylim(0, 105)

ax = axes[1]
ax.plot(RATES, mean_recv, color=BLUE,  marker='o', ms=8, lw=2, label='Reported')
ax.plot(RATES, mean_eff,  color=GREEN, marker='s', ms=8, lw=2, label='Effective')
ax.fill_between(RATES, mean_recv, mean_eff, alpha=0.12, color=ORANGE, label='Loss gap')
ax.set_xscale('log'); ax.set_xticks(RATES)
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.set_xlabel('Target Rate (Mbps)'); ax.set_ylabel('Throughput (Mbps)')
ax.set_title('Reported vs Effective Throughput'); ax.legend()
plt.suptitle('Loss vs Throughput Analysis', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Per-UE Throughput at 100 Mbps ───────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(18, 9))
sub = udp_df[udp_df.rate_target == 100].sort_values('ue_id')
x = np.arange(len(sub))

ax = axes[0]
ax.bar(x, sub['tput_mbps'], color=BLUE, edgecolor='white', width=0.75)
ax.axhline(100, color=ORANGE, ls='--', lw=1.3, label='Target 100 Mbps')
ax.set_xticks(x); ax.set_xticklabels([f'UE{u}' for u in sub['ue_id']], rotation=45, ha='right', fontsize=7.5)
ax.set_ylabel('Reported Tput (Mbps)'); ax.set_title('Reported Throughput @ 100 Mbps — All UEs'); ax.legend()

ax = axes[1]
colors_ = [GREEN if e > 0.5 else RED for e in sub['effective_mbps']]
ax.bar(x, sub['effective_mbps'], color=colors_, edgecolor='white', width=0.75)
ax.set_xticks(x); ax.set_xticklabels([f'UE{u}' for u in sub['ue_id']], rotation=45, ha='right', fontsize=7.5)
ax.set_ylabel('Effective Tput (Mbps)'); ax.set_title('Effective Throughput @ 100 Mbps — Reported × (1−Loss%)')
plt.tight_layout(); plt.show()

In [ ]:
# ── Box plots: distribution at each rate ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Distribution at Each Rate (all UEs)', fontsize=13, fontweight='bold')
metrics = [('tput_mbps','Reported Tput (Mbps)',BLUE), ('effective_mbps','Effective Tput (Mbps)',GREEN), ('loss_pct','Loss %',RED)]
for (col, ylabel, c), ax in zip(metrics, axes):
    data = [udp_df[udp_df.rate_target==r][col].dropna().tolist() for r in RATES]
    bp = ax.boxplot(data, patch_artist=True, tick_labels=[f'{r}M' for r in RATES], widths=0.5)
    for patch in bp['boxes']: patch.set_facecolor(c); patch.set_alpha(0.65)
    for m in bp['medians']: m.set_color('black'); m.set_linewidth(2.5)
    ax.set_xlabel('Target Rate'); ax.set_ylabel(ylabel); ax.set_title(ylabel)
plt.tight_layout(); plt.show()

In [ ]:
# ── Scatter: Effective Tput vs Loss, coloured by rate ───────────────────
fig, ax = plt.subplots(figsize=(11, 6))
cmap = plt.cm.viridis
norm = mcolors.LogNorm(vmin=1, vmax=500)
for rate in RATES:
    sub = udp_df[udp_df.rate_target == rate]
    ax.scatter(sub['effective_mbps'], sub['loss_pct'],
               color=cmap(norm(rate)), alpha=0.65, s=45, edgecolors='none', label=f'{rate}M')
ax.set_xlabel('Effective Throughput (Mbps)')
ax.set_ylabel('Packet Loss %')
ax.set_title('Effective Throughput vs Packet Loss\nEach dot = one UE (colour = target rate)')
ax.legend(title='Target Rate', ncol=3, fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# ── Effective Throughput Heatmap (UE x Rate) ────────────────────────────
pivot = udp_df.pivot_table(index='ue_id', columns='rate_target',
                            values='effective_mbps', aggfunc='mean').reindex(columns=RATES)
fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(pivot.values, aspect='auto', cmap='YlGn', vmin=0, vmax=25)
ax.set_xticks(range(len(RATES))); ax.set_xticklabels([f'{r}M' for r in RATES])
ax.set_yticks(range(len(pivot))); ax.set_yticklabels([f'UE{u}' for u in pivot.index], fontsize=7)
ax.set_xlabel('Target Rate'); ax.set_ylabel('UE')
ax.set_title('Effective Throughput Heatmap — All UEs × All Rates (capped 25 Mbps)')
plt.colorbar(im, ax=ax, shrink=0.6, label='Effective Tput (Mbps)')
plt.tight_layout(); plt.show()

In [ ]:
# ── ICMP Latency per UE ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(18, 9))
ping = ping_df.sort_values('ue_id')
x = np.arange(len(ping)); w = 0.28

ax = axes[0]
ax.bar(x, ping['ping_avg_ms'], color=BLUE, edgecolor='white', width=0.7)
ax.errorbar(x, ping['ping_avg_ms'], yerr=ping['ping_mdev_ms'],
            fmt='none', ecolor=ORANGE, capsize=3, lw=1.5, label='±jitter (mdev)')
ax.set_xticks(x); ax.set_xticklabels([f'UE{u}' for u in ping['ue_id']], rotation=45, ha='right', fontsize=7.5)
ax.set_ylabel('Avg RTT (ms)'); ax.set_title('ICMP Latency — Idle (bars=avg, error=±jitter)'); ax.legend()

ax = axes[1]
ax.bar(x-w, ping['ping_min_ms'], width=w, color=GREEN, edgecolor='white', label='Min')
ax.bar(x,   ping['ping_avg_ms'], width=w, color=BLUE,  edgecolor='white', label='Avg')
ax.bar(x+w, ping['ping_max_ms'], width=w, color=RED,   edgecolor='white', label='Max')
ax.set_xticks(x); ax.set_xticklabels([f'UE{u}' for u in ping['ue_id']], rotation=45, ha='right', fontsize=7.5)
ax.set_ylabel('RTT (ms)'); ax.set_title('Min / Avg / Max RTT per UE'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ── RAN Radio Metrics ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('RAN Radio Metrics — 50 UE Slots', fontsize=13, fontweight='bold')
ran = ran_df.sort_values('ue_id').reset_index(drop=True)
x = np.arange(len(ran)); labels = [f'UE{u}' for u in ran['ue_id']]
plots = [('pucch_snr','PUCCH SNR (dB)',BLUE,(0,130)),
         ('cqi','CQI (0-15)',GREEN,(0,17)),
         ('prb_util_pct','PRB Util %',PURPLE,(0,100)),
         ('pusch_snr','PUSCH SNR (dB)',ORANGE,(0,130)),
         ('pusch_mcs','UL MCS',RED,(0,30)),
         ('pdsch_mcs','DL MCS','#06b6d4',(0,30))]
for (col, title, c, ylim), ax in zip(plots, axes.flat):
    ax.bar(x, ran[col].fillna(0), color=c, edgecolor='white', width=0.75)
    ax.set_xticks(x[::5]); ax.set_xticklabels(labels[::5], rotation=45, ha='right', fontsize=7)
    ax.set_ylim(*ylim); ax.set_title(title)
plt.tight_layout(); plt.show()

In [ ]:
# ── Summary Statistics ───────────────────────────────────────────────────
print('UDP by rate:')
display(udp_df.groupby('rate_target')[['tput_mbps','effective_mbps','loss_pct','jitter_ms']].agg(['mean','min','max']).round(2))
print('\nICMP Latency:')
display(ping_df[['ping_avg_ms','ping_min_ms','ping_max_ms','ping_mdev_ms']].describe().round(1))
print('\nRAN Metrics:')
display(ran_df[['pucch_snr','cqi','prb_util_pct','pusch_mcs','pdsch_mcs']].describe().round(2))